# Session 2 — From a chatbot to an LLM-based agent

**DAT 5668 · Module 1, Session 2 · Practice (about 80 minutes)**

**Family this session: LLM-based agent.** You will build the tool loop from Session 1. The RL-based agent starts Session 2 of Week 2.

![LLM-based agent tool loop](../../assets/llm-tool-loop.png)

## Run of show
- 10 min — Recap Session 1's loop. Entry question: why did ticket B differ across families?
- 55 min — Practice below. Complete every `TODO`.
- 15 min — Debrief: which failures required a tool, and which still need a person?

## Business setting
Service desk at a retailer. Instructional tickets, not firm data. You may refund a **damaged** item only when the order is **30 days old or newer**. The refund amount must come from the order record.

A chatbot that "sounds right" is not acceptable. The agent has to act through tools.


In [ ]:
ORDERS = {
    "1001": {"age_days": 12, "total": 80, "note": "cracked mug"},
    "1002": {"age_days": 10, "total": 40, "note": "changed mind"},
    "1003": {"age_days": 45, "total": 200, "note": "cracked screen"},
    "1004": {"age_days": 4, "total": 15, "note": "missing part"},
}

CASES = []


def lookup_order(order_id):
    """Tool: read the system of record. Returns None if the id is unknown."""
    return ORDERS.get(str(order_id))


def lookup_policy(note):
    """Tool: the rule, not the model's opinion."""
    note = note.lower()
    if "crack" in note or "missing" in note:
        return {"eligible_reason": True, "max_age_days": 30}
    return {"eligible_reason": False, "max_age_days": 0}


def open_case(order_id, reason):
    """Tool: the action that leaves a trail."""
    CASES.append({"order_id": order_id, "reason": reason})
    return {"case_id": len(CASES), "reason": reason}


def chatbot_reply(order_id):
    """What 'just prompt it' does: a fluent answer with no tool and no record."""
    return {
        "action": "refund",
        "amount": 999,  # invented
        "reason": "Sorry about that, I have issued a full refund.",
        "case_id": None,
    }


print("Chatbot on 1003 (should NOT be trusted):", chatbot_reply("1003"))


## Your work

Complete `decide`. It must:

1. Observe the order with `lookup_order`. Unknown ids escalate.
2. Observe the rule with `lookup_policy` on the order note.
3. Act: `refund` only when the reason is eligible and `age_days <= max_age_days`. Otherwise `refuse`.
4. If the order is missing, act `escalate` and call `open_case`.
5. The refund amount equals `order["total"]`. Never hard-code 999.

Return a dict with keys `action`, `amount`, `reason`, `case_id`.


In [ ]:
def decide(order_id):
    """TODO: implement the agent loop using the tools above."""
    # TODO: observe, decide, act, and return feedback the desk can audit.
    return {"action": "escalate", "amount": 0, "reason": "TODO", "case_id": None}


def check_decide():
    CASES.clear()
    expected = {
        "1001": ("refund", 80),
        "1002": ("refuse", 0),
        "1003": ("refuse", 0),
        "1004": ("refund", 15),
        "9999": ("escalate", 0),
    }
    ok = True
    for order_id, (action, amount) in expected.items():
        result = decide(order_id)
        match = result["action"] == action and result["amount"] == amount
        ok = ok and match
        flag = "PASS" if match else "NOT YET"
        print(f"{flag}  order {order_id}: got {result['action']} ${result['amount']}  expected {action} ${amount}")
    if ok and any(case["order_id"] == "9999" for case in CASES):
        print("PASS  unknown order opened a case")
    elif ok:
        print("NOT YET  escalate must call open_case")
        ok = False
    print("Ready for debrief." if ok else "Keep going — the chatbot cell is the failure mode to beat.")
    return ok


check_decide()


## Written debrief (5 minutes, in the notebook or on paper)

Answer in a markdown cell you add, or in class notes:

1. Which ticket would the chatbot have gotten expensively wrong, and why?
2. Which action still needs a human even after the tools are correct?
3. Name one business process at a firm where this tool pattern fits, and one where it does not.


## Fill this with me

Run the chatbot cell first. Say the invented dollar amount. Then we write `decide` together until every line says PASS.


In [ ]:
# FILL WITH THE CLASS. What amount did the chatbot invent for order 1003?
invented_amount = None
print("Chatbot invented", invented_amount, ".")
print("If that number is not the order total, the tools have to own the amount.")
